# Laboratorio práctico de ETL — Kaismart Solutions S.A.S.
**Universidad Autónoma de Occidente** · Metodología **Medallion** (Bronze → Silver → Gold)

Autores: ver `config.yaml`.  Fuentes: MySQL `clientes.ventas` (5.000 registros) y `kaismart_eventos_logisticos.xlsx` (50.000 eventos), unidas por `pedido_id`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import sys, os
sys.path.insert(0, os.path.abspath(".."))      # para importar el paquete etl/ desde notebooks/
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from etl.config import cargar_config, ruta, RAIZ
from etl import profiling as pf
from etl.transform import Decisiones, silver_ventas, silver_logistica, gold_pedidos, ORDEN_ESTADOS

pd.set_option("display.max_columns", 50); pd.set_option("display.width", 220)
cfg = cargar_config()
print("Autores:", cfg["autores"])

## PARTE 1 — Extracción desde MySQL (Bronze)

In [ ]:
# 1) librerías  2) conexión  3-4) consulta de toda la tabla  5) DataFrame  7) cierre de la conexión
m = cfg["mysql"]
conn = None
try:
    import mysql.connector
    conn = mysql.connector.connect(host=m["host"], user=m["user"], password=m["password"], database=m["database"])
    df_ventas = pd.read_sql("SELECT * FROM ventas", conn)
except Exception as e:                      # plan B opcional: CSV exportado (config.yaml -> ventas_csv_respaldo)
    if not cfg.get("ventas_csv_respaldo"): raise
    print("MySQL no disponible, usando respaldo:", e); df_ventas = pd.read_csv(cfg["ventas_csv_respaldo"])
finally:
    if conn is not None and conn.is_connected():
        conn.close(); print("Conexión cerrada")

In [ ]:
# 6) comprobación
print(df_ventas.shape); print(list(df_ventas.columns)); display(df_ventas.head()); display(df_ventas.sample(5, random_state=42))

## PARTE 2 — Extracción desde Excel (Bronze)

In [ ]:
df_logistica = pd.read_excel(RAIZ / cfg["excel"]["ruta"], sheet_name=cfg["excel"]["hoja"])
print(df_logistica.shape); print(list(df_logistica.columns)); display(df_logistica.head()); display(df_logistica.sample(5, random_state=42))

In [ ]:
# Bronze: copia fiel e intacta de las dos fuentes
df_ventas.to_csv(ruta(cfg, "bronze", "ventas_raw.csv"), index=False)
df_logistica.to_csv(ruta(cfg, "bronze", "logistica_raw.csv"), index=False)

## PARTE 3 — Comprensión inicial

### 3.1 df_ventas

In [ ]:
print("Registros:", len(df_ventas), "| Variables:", df_ventas.shape[1])
display(pf.estructura(df_ventas))
roles_v = pf.clasificar_columnas(df_ventas); print(roles_v)
display(pf.rango_fechas(df_ventas))

**¿Qué representa una fila de `df_ventas`?** Una **venta** (una transacción de un cliente): identifica el pedido, la ciudad, el canal, la categoría del producto, la cantidad, los valores monetarios (precio, bruto, descuento y neto) y, si existe, la calificación del cliente. Es una granularidad de **un pedido por fila** (se valida con `nunique` en la Parte 4).

### 3.2 df_logistica

In [ ]:
print("Registros:", len(df_logistica), "| Variables:", df_logistica.shape[1])
display(pf.estructura(df_logistica))
roles_l = pf.clasificar_columnas(df_logistica); print(roles_l)
display(pf.rango_fechas(df_logistica))

**¿Qué representa una fila de `df_logistica`?** Un **evento del ciclo de vida de un pedido** (pedido recibido, pago aprobado, … entregado). Cada pedido tiene ~10 eventos, por eso hay 50.000 filas para 5.000 pedidos (relación **1 venta : N eventos**).

**Observaciones de calidad (Parte 3, punto 10):**
- Hay una columna `Unnamed: 13` con una fórmula residual de Excel (`=AI("")`) en 1 sola fila → columna basura.
- `fecha_evento` y `fecha_prometida_entrega` se leen como **texto** (`str`), no como fecha.
- Rango de fechas de eventos: **2026-01-01 → 2026-07-03**.
- Identificadores: `evento_id`, `pedido_id`, `numero_guia`. Fechas/tiempos: `fecha_evento`, `fecha_prometida_entrega`, `tiempo_etapa_horas`. Numéricas: `tiempo_etapa_horas`, `costo_envio`. Categóricas: `estado_evento`, `centro_logistico`, `ciudad_destino`, `transportadora`, `incidencia`, `observacion`.

## PARTE 4 — Perfil inicial de calidad (sin modificar nada)

### 4.1 Valores nulos

In [ ]:
nulos_v = pf.perfil_nulos(df_ventas); nulos_l = pf.perfil_nulos(df_logistica)
display(nulos_v); display(nulos_l)

**Interpretación de nulos en logística**
| Variable | % nulo | ¿Normal o problema? |
|---|---|---|
| `Unnamed: 13` | 100% | Problema: columna basura de Excel |
| `incidencia` | 98,98% | **Normal**: solo ~1% de eventos tiene novedad |
| `observacion` | 96,55% | **Normal** en su mayoría: solo se anota cuando hay algo que decir |
| `transportadora`, `numero_guia` | ~60% | **Normal** en los 6 estados previos al despacho (60% de los eventos); **problema** en ~88 filas posteriores al despacho |
| `tiempo_etapa_horas` | 10% | **Normal**: coincide con el primer evento de cada pedido ("Pedido recibido"), que no tiene etapa anterior |
| `centro_logistico`, `ciudad_destino`, `fecha_prometida_entrega`, `fecha_evento` | 0,09%–0,18% | **Problema de calidad**: son atributos que siempre deberían existir |

### 4.2 Valores únicos

In [ ]:
display(pf.perfil_unicos(df_ventas)); display(pf.perfil_unicos(df_logistica))

Lectura: `pedido_id` tiene 5.000 valores únicos en logística (1 pedido → ~10 eventos); `evento_id` debería ser único en 50.000 pero tiene 49.900 → **100 ids repetidos**; `numero_guia` tiene 4.999 valores (uno por pedido despachado). Baja cardinalidad: `estado_evento` (10), `centro_logistico` (6), `ciudad_destino` (6), `transportadora` (5), `incidencia` (6), `costo_envio` (12). Alta cardinalidad: `evento_id`, `fecha_evento`, `pedido_id`, `numero_guia`.

### 4.3 Duplicados

In [ ]:
print("ventas   :", pf.perfil_duplicados(df_ventas, ["id_venta", "pedido_id"]))
print("logística:", pf.perfil_duplicados(df_logistica, ["evento_id", "pedido_id", "numero_guia"]))

En logística hay **99 filas completamente duplicadas** (0,2%) y **100 `evento_id` repetidos**: 99 son copias exactas y 1 es el mismo evento con y sin `centro_logistico`. Que `pedido_id` y `numero_guia` se repitan **no es error** (un pedido tiene ~10 eventos).

### 4.4 Tipos de datos

In [ ]:
display(pf.perfil_tipos(df_ventas)); display(pf.perfil_tipos(df_logistica))

Columnas a revisar antes de analizar (logística): `fecha_evento` y `fecha_prometida_entrega` (texto → datetime); `pedido_id` y `numero_guia` (texto, correcto como identificador); `evento_id` (int, correcto); `costo_envio` (int, monetaria; correcta en pesos); `tiempo_etapa_horas` (float, correcta). En ventas: revisar que las fechas sean datetime y los valores monetarios sean numéricos (no texto).

## PARTE 5 — Estadísticos descriptivos (sin integrar)

### 5.1 Variables numéricas

In [ ]:
display(pf.describe_numericas(df_ventas)); display(pf.describe_numericas(df_logistica))

### 5.2 Variables categóricas — ventas

In [ ]:
for col in [c for c in ["ciudad", "canal", "categoria"] if c in df_ventas.columns]:
    print(pf.resumen_categorica(df_ventas, col)); display(pf.frecuencias(df_ventas, col))
if "cantidad" in df_ventas: display(pf.frecuencias(df_ventas, "cantidad"))
for col in [c for c in ["precio_unitario", "valor_bruto", "valor_descuento", "valor_neto"] if c in df_ventas.columns]:
    print(col); display(df_ventas[col].describe().to_frame().T)
if "calificacion_cliente" in df_ventas:
    display(df_ventas["calificacion_cliente"].value_counts(dropna=True).sort_index())

### 5.2 Variables categóricas — logística

In [ ]:
for col in ["estado_evento", "ciudad_destino", "transportadora", "incidencia"]:
    print(pf.resumen_categorica(df_logistica, col)); display(pf.frecuencias(df_logistica, col, dropna=False))
display(df_logistica["tiempo_etapa_horas"].describe().to_frame().T); display(df_logistica["costo_envio"].describe().to_frame().T)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
df_logistica["estado_evento"].value_counts().reindex(ORDEN_ESTADOS).plot.barh(ax=ax[0], title="Eventos por estado"); ax[0].invert_yaxis()
sns.histplot(df_logistica["tiempo_etapa_horas"].dropna(), bins=40, ax=ax[1]); ax[1].set_title("tiempo_etapa_horas")
df_logistica["costo_envio"].value_counts().sort_index().plot.bar(ax=ax[2], title="costo_envio"); plt.tight_layout(); plt.show()

## PARTE 6 — Preguntas de negocio (sobre los datos originales)

In [ ]:
fcol = next((c for c in df_ventas.columns if "fecha" in c.lower()), None)
V, L = df_ventas, df_logistica
print("P1. ¿Qué ciudad vende más (valor neto)?");               display(V.groupby("ciudad")["valor_neto"].sum().sort_values(ascending=False))
print("P2. ¿Qué canal genera más ingresos y cuál tiene mayor ticket promedio?"); display(V.groupby("canal")["valor_neto"].agg(["sum", "mean", "count"]))
print("P3. ¿Qué categoría tiene mayor valor neto?");            display(V.groupby("categoria")["valor_neto"].sum().sort_values(ascending=False))
print("P4. ¿Qué % del valor bruto se pierde en descuentos, por categoría?"); display((V.groupby("categoria")["valor_descuento"].sum() / V.groupby("categoria")["valor_bruto"].sum() * 100).round(2))
print("P5. ¿Cuál es la calificación promedio por canal?");      display(V.groupby("canal")["calificacion_cliente"].agg(["mean", "count"]))
if fcol: 
    print("P6. ¿Cómo evolucionan las ventas por mes?");         display(V.assign(m=pd.to_datetime(V[fcol], errors="coerce").dt.to_period("M")).groupby("m")["valor_neto"].agg(["sum", "count"]))
print("P7. ¿Qué transportadora acumula más incidencias (y qué % de sus eventos)?")
d = L.dropna(subset=["transportadora"]).assign(inc=L["incidencia"].notna()); display(d.groupby("transportadora")["inc"].agg(["sum", "mean"]).sort_values("sum", ascending=False))
print("P8. ¿Qué etapa del proceso tarda más en promedio?");     display(L.groupby("estado_evento")["tiempo_etapa_horas"].mean().sort_values(ascending=False).round(2))
print("P9. ¿Cuál es el costo de envío promedio por ciudad?");   display(L.drop_duplicates().groupby("ciudad_destino")["costo_envio"].mean().round(0).sort_values(ascending=False))
print("P10. ¿Qué % de pedidos se entregó después de la fecha prometida?")
e = L.drop_duplicates(); e = e[e["estado_evento"] == "Entregado"]
tarde = pd.to_datetime(e["fecha_evento"]) > pd.to_datetime(e["fecha_prometida_entrega"]); print(f"{tarde.mean()*100:.2f}% de {len(e)} entregas")

## PARTE 7 — Transformación (Medallion)
- **Bronze**: datos crudos tal como llegaron (`data/bronze/`).
- **Silver**: datos limpios y estandarizados → `df_ventas_transformado`, `df_logistica_transformado` (`data/silver/`).
- **Gold**: tabla integrada por `pedido_id` lista para análisis (`data/gold/pedidos_gold.csv`).

Cada decisión de imputación queda explicada en el DataFrame `decisiones` y en `reports/decisiones_transformacion.csv`.

In [ ]:
dec = Decisiones()
df_ventas_transformado = silver_ventas(df_ventas, dec)
df_logistica_transformado = silver_logistica(df_logistica, dec)
df_ventas_transformado.to_csv(ruta(cfg, "silver", "ventas_silver.csv"), index=False)
df_logistica_transformado.to_csv(ruta(cfg, "silver", "logistica_silver.csv"), index=False)
df_gold = gold_pedidos(df_ventas_transformado, df_logistica_transformado, dec)
df_gold.to_csv(ruta(cfg, "gold", "pedidos_gold.csv"), index=False)
decisiones = dec.df(); decisiones.to_csv(ruta(cfg, "reportes", "decisiones_transformacion.csv"), index=False, encoding="utf-8-sig")
print("Silver ventas:", df_ventas_transformado.shape, "| Silver logística:", df_logistica_transformado.shape, "| Gold:", df_gold.shape)

### 7.1 Decisiones de limpieza (problema → cuantificación → acción → justificación)

In [ ]:
with pd.option_context("display.max_colwidth", 110, "display.max_rows", 200):
    display(decisiones)

### 7.2 Antes vs después

In [ ]:
comp = pd.DataFrame({
    "filas_antes": [len(df_ventas), len(df_logistica)], "filas_despues": [len(df_ventas_transformado), len(df_logistica_transformado)],
    "nulos_antes": [int(df_ventas.isna().sum().sum()), int(df_logistica.isna().sum().sum())],
    "nulos_despues": [int(df_ventas_transformado.isna().sum().sum()), int(df_logistica_transformado.isna().sum().sum())],
    "duplicados_antes": [int(df_ventas.duplicated().sum()), int(df_logistica.duplicated().sum())],
    "duplicados_despues": [int(df_ventas_transformado.duplicated().sum()), int(df_logistica_transformado.duplicated().sum())],
}, index=["ventas", "logistica"]); display(comp)
display(pf.perfil_nulos(df_logistica_transformado).query("nulos > 0"))

**Nulos que se mantienen a propósito (tienen interpretación válida):** `transportadora`/`numero_guia` antes del despacho, `tiempo_etapa_horas` en "Pedido recibido" y `calificacion_cliente` cuando el cliente no calificó.

In [ ]:
print("Pedidos entregados a tiempo vs tarde (Gold):"); display(df_gold["entrega_a_tiempo"].value_counts(dropna=False))
print("Pedidos con ciclo incompleto (<10 estados):", (~df_gold["pedido_completo"]).sum())
display(df_gold.head())

## PARTE 8 — Automatización con `schedule`
El orquestador está en `scheduler.py`; ejecuta `etl.pipeline.ejecutar_pipeline()` todos los días a la hora definida en `config.yaml`, registra todo en `logs/etl.log` y, si una ejecución falla, registra el error pero **el orquestador sigue vivo**.

```bash
python scheduler.py --una-vez   # prueba: corre el ETL una vez
python scheduler.py             # queda programado (diario, 02:00)
```

In [ ]:
# Demostración: misma función que el scheduler
import schedule
from etl.pipeline import ejecutar_pipeline
schedule.clear(); schedule.every().day.at(cfg["scheduler"]["hora_diaria"]).do(lambda: ejecutar_pipeline(cfg))
print("Próxima ejecución programada:", schedule.next_run()); schedule.clear()

## CONCLUSIONES

In [ ]:
v, vt = df_ventas, df_ventas_transformado
print("HALLAZGOS df_ventas")
print(f"1. Estructura: {v.shape[0]:,} registros y {v.shape[1]} variables; granularidad de una venta por fila.")
print(f"2. Duplicados: {int(v.duplicated().sum())} filas completas duplicadas y {int(v['id_venta'].duplicated().sum()) if 'id_venta' in v else 'N/A'} id_venta repetidos.")
top = pf.perfil_nulos(v).query("nulos > 0").head(3)
print(f"3. Nulos: " + (", ".join(f"{i} ({r.porcentaje}%)" for i, r in top.iterrows()) if len(top) else "sin nulos") + ".")
for col in [c for c in ["ciudad", "canal", "categoria"] if c in v]:
    print(f"4. {col}: {v[col].nunique()} categorías sin estandarizar -> {vt[col].nunique()} tras limpieza; más frecuente: {vt[col].mode()[0]}.")
if "valor_neto" in v: print(f"5. valor_neto: media {v['valor_neto'].mean():,.0f}, mediana {v['valor_neto'].median():,.0f}, máx {v['valor_neto'].max():,.0f} (distribución sesgada a la derecha si media > mediana).")
if "calificacion_cliente" in v: print(f"6. calificacion_cliente: {v['calificacion_cliente'].isna().mean()*100:.1f}% sin calificación (se mantuvo nulo: es información válida).")
print(f"7. Transformación: {len(v)-len(vt)} filas eliminadas; flags de inconsistencia -> bruto: {int(vt.get('flag_bruto_inconsistente', pd.Series(dtype=int)).sum())}, neto: {int(vt.get('flag_neto_inconsistente', pd.Series(dtype=int)).sum())}.")

**HALLAZGOS df_logistica**
1. **Estructura:** 50.000 eventos × 14 columnas para 5.000 pedidos (~10 eventos por pedido, 10 estados del ciclo). Rango: 2026-01-01 a 2026-07-03.
2. **Columna basura:** `Unnamed: 13` con una fórmula residual de Excel (`=AI("")`) en una sola fila → eliminada en Silver.
3. **Duplicados:** 99 filas completamente duplicadas (0,2%) y 100 `evento_id` repetidos (99 copias exactas + 1 con datos incompletos) → 49.900 eventos únicos. Consecuencia: 98 pedidos quedaron con el ciclo incompleto (8 o 9 de 10 estados) y 10 sin evento "Entregado".
4. **Nulos:** `incidencia` 98,98% y `observacion` 96,55% (normales: no hubo novedad); `transportadora`/`numero_guia` ~60% (normales antes del despacho, ~88 filas anómalas después); `tiempo_etapa_horas` 10% (= primer evento de cada pedido); `centro_logistico` 0,18%, `fecha_evento` 0,13%, `fecha_prometida_entrega` 0,11% y `ciudad_destino` 0,09% son errores recuperables.
5. **Tipos:** las dos columnas de fecha se leen como texto; los identificadores y el resto de tipos son coherentes.
6. **Cardinalidad:** `costo_envio` solo tiene 12 valores y **no depende de la ciudad** (promedios de ~13.500 en las 6 ciudades). Los valores terminados en 90 (9.790, 10.890, 13.090, 15.290, 17.490, 20.790) son exactamente +10% sobre las tarifas base → ~25% de los pedidos tiene un recargo.
7. **Negocio:** solo ~1% de los eventos tiene incidencia (511; la más frecuente "Error de clasificación", 98) y ~17% de los pedidos se entrega después de la fecha prometida. La etapa más lenta es "En tránsito" (≈9 h promedio por evento).

**HALLAZGOS DE LA TRANSFORMACIÓN**
- **Bronze** conserva los datos originales; **Silver** deja 0 duplicados, fechas en `datetime` y categorías estandarizadas; **Gold** integra ventas + logística por `pedido_id` (una fila por pedido) con métricas nuevas: `n_eventos`, `horas_totales_proceso`, `estado_final`, `fecha_entrega`, `entrega_a_tiempo`, `dias_hasta_entrega`, `n_incidencias`, `pedido_completo`.
- Las 65 `fecha_evento` nulas se **reconstruyeron** con la fecha del evento anterior + `tiempo_etapa_horas` (en lugar de usar media/mediana, que habrían inventado fechas).
- Los atributos fijos por pedido (ciudad, CEDI, fecha prometida, transportadora, guía) se recuperaron desde otros eventos del **mismo pedido**; los nulos con significado de negocio se conservaron o se rotularon (`SIN INCIDENCIA`, `SIN OBSERVACIÓN`) y **no se imputó nada de forma automática**.